# 02b — Gain and amplitude dynamics

Reviews registered Q metrics, extraction support, observed ranges, and saved distributions.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main outputs: `outputs/02_features/gain_dynamics/tables/metric_summary.csv` and `figures/metric_distributions.*`.

Decision required only if extraction errors occur, a metric is entirely missing, or support/status distributions are scientifically implausible. Do not tune thresholds after examining clinical or human-label associations.

In [ ]:
from paper1_qc.registry import metric_registry_frame

FAMILY = 'gain_dynamics'
PREFIX = 'qgain'
STAGE, FIGURES, TABLES = stage_directories(Path("02_features") / FAMILY)
registry = metric_registry_frame()
family_registry = registry.loc[registry["family"].eq(FAMILY)].copy()
display(family_registry)
save_table(family_registry, TABLES, "metric_registry")

metrics = read_table(OUTPUT / "02_features" / "bamboo_q_metrics")
errors = read_table(OUTPUT / "02_features" / "feature_extraction_errors")
features = [feature for feature in family_registry["feature"] if feature in metrics.columns]

summary_rows = []
for feature in features:
    values = pd.to_numeric(metrics[feature], errors="coerce")
    summary_rows.append({
        "feature": feature,
        "recordings": len(values),
        "nonmissing": int(values.notna().sum()),
        "missing_fraction": float(values.isna().mean()),
        "zero_fraction_nonmissing": float(values.dropna().eq(0).mean()) if values.notna().any() else np.nan,
        "median": values.median(),
        "q25": values.quantile(0.25),
        "q75": values.quantile(0.75),
        "minimum": values.min(),
        "maximum": values.max(),
    })
metric_summary = pd.DataFrame(summary_rows)
status_columns = [column for column in metrics if column.startswith(PREFIX) and column.endswith("_status")]
status_summary = (
    metrics[status_columns].melt(var_name="status_field", value_name="status")
    .groupby(["status_field", "status"], dropna=False).size()
    .rename("recordings").reset_index()
    if status_columns else pd.DataFrame()
)
save_table(metric_summary, TABLES, "metric_summary")
save_table(status_summary, TABLES, "status_summary")
save_table(errors, TABLES, "extraction_errors")
display(metric_summary)
display(status_summary)

ncols = 3
nrows = max(1, int(np.ceil(len(features) / ncols)))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.8 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, feature in zip(axes, features):
    sns.histplot(pd.to_numeric(metrics[feature], errors="coerce"), bins=30, ax=ax, color="#4C78A8")
    ax.set_title(feature, fontsize=9)
    ax.set_xlabel("")
for ax in axes[len(features):]:
    ax.set_axis_off()
fig.suptitle(f"{FAMILY.replace('_', ' ').title()} — observed metric distributions")
fig.tight_layout()
save_figure(fig, FIGURES, "metric_distributions")
plt.show()

blocking = []
if not errors.empty:
    blocking.append(f"{len(errors)} extraction errors require investigation.")
if metric_summary.empty:
    blocking.append("No registered metrics were found.")
else:
    all_missing = metric_summary.loc[metric_summary["nonmissing"].eq(0), "feature"].tolist()
    if all_missing:
        blocking.append("Entirely missing metrics: " + ", ".join(all_missing))

feature_stage_ready = stage_gate(
    FAMILY.replace("_", " ").title(),
    not blocking,
    blocking,
    "Proceed to the next family notebook if PASS; otherwise inspect saved errors/support tables and rerun.",
)